# Часть 2. Обогащение данных из стороннего датасета

В этом ноутбуке мы загружаем очищенный датасет `netflix_clean_1.csv`.
И обогащаем его информацией о жанрах, бюджете, выручке, хронометраже, данные об аудитории и оценке из внешних источников.

Результат этой части - это обогащенный датасет `netflix_enriched_2.csv`, который используется в последующих частях анализа.

#### Выполнила Марокина Татьяна


## Шаг 1. Импорт библиотек и чтение датасетов


In [230]:
import pandas as pd
import numpy as np
import ast


### 1.1 Чтение "очищенного" датасета Netflix


In [231]:
df_cleared = pd.read_csv("netflix_clean_1.csv")
print(df_cleared.shape)
df_cleared.head(3)

(499, 4)


,title,rating,release_year,title_lower
0,White Chicks,PG-13,2004,white chicks
1,Lucky Number Slevin,R,2006,lucky number slevin
2,Grey's Anatomy,TV-14,2016,grey's anatomy


### 1.2 Чтение стороннего датасета для обогащения `TMDB_IMDB_Movies_Dataset.csv`

Датасет взят с [kaggle](https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset?select=TMDB++IMDB+Movies+Dataset.csv). Датаесет содержит 437246 строк и 29 признаков.

В этом датасете представлены следующие признаки, которыми мы планируем обогатить исходный датасет:

- рейтинги и оценки IMDB `averageRating`, `numVotes`, id для парсинга `tconst`
- данные по бюджету и выручке `revenue`, `budget`
- информация о продолжительности, жанрах, популярность `runtime`, `genres`, `popularity`


In [232]:
df_tmdb_imdb_movies = pd.read_csv("TMDB_IMDB_Movies_Dataset.csv")
print(df_tmdb_imdb_movies.shape)
df_tmdb_imdb_movies.head(3)

(437246, 29)


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2813028,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2523452,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3163410,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."


In [233]:
df_tmdb_imdb_movies.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'tconst', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='str')

### 1.3 Чтение стороннего датасета `title.basics.tsv`

Датасет взят с [imdb.com](https://developer.imdb.com/non-commercial-datasets/). Датасет содержит 12475319 строк и 9 признаков.

Загружаем только 4 колонrb — достаточно для точного поиска `imdb_id` по названию на этапе слияния. Полный датасет с жанрами и хронометражем будет использован в шаге 4.


In [234]:
df_imdb_ids = pd.read_csv(
    "title.basics.tsv",
    sep="\t",
    na_values=["\\N"],
    usecols=["tconst", "primaryTitle", "startYear", "endYear"],
    low_memory=False,
)

print(df_imdb_ids.shape)
df_imdb_ids.head(3)

(12475319, 4)


,tconst,primaryTitle,startYear,endYear
0,tt0000001,Carmencita,1894.0,NaN
1,tt0000002,Le clown et ses chiens,1892.0,NaN
2,tt0000003,Poor Pierrot,1892.0,NaN


## Шаг 2. Чистка датасетов и подготовка к слиянию


### 2.1 Чистка `df_tmdb_imdb_movies`

Приготовим к слиянию датасет. Убираем лишние признаки, которые не планируем использовать в анализе


In [235]:
columns = [
    "title",
    "tconst",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "release_date",
    "averageRating",
    "numVotes",
    "popularity",
]
df_tmdb_imdb_movies = df_tmdb_imdb_movies[columns].copy()

Для избежания ошибок в слиянии датасетов необходимо избавиться полностью от строк без названия. Все остальные признаки можно пока оставить как есть, поскольку избавляться от пропусков, если они есть, мы должны будем после слияния. Пока-что мы просто пытаемся обогатить исходный датасет информацией.

- Избавимся также от всех пропусков по полю `title` и дублей по полю `title` + `release_year`. Для целей обогащения такие строки попросту не нужны.
- Дополнительно подготовим столбец с годом `release_year`, для дальнейшего слияния в паре с названием `title`.
- переименуем столбцы для единообразия и разделения источников (для рейтинга)
- Приведем к нужному формату


In [236]:
df_tmdb_imdb_movies = df_tmdb_imdb_movies.dropna(subset=["title"]).copy()
df_tmdb_imdb_movies["title"] = df_tmdb_imdb_movies["title"].str.lower()

df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(columns={"tconst": "imdb_id"})
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"averageRating": "vote_average_imdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"numVotes": "vote_count_imdb"}
)

df_tmdb_imdb_movies["release_year"] = pd.to_datetime(
    df_tmdb_imdb_movies["release_date"], errors="coerce"
).dt.year.astype("Int64")

df_tmdb_imdb_movies = (
    df_tmdb_imdb_movies.drop(columns=["release_date"])
    .drop_duplicates(subset=["title", "release_year"])
    .reset_index(drop=True)
)

print(df_tmdb_imdb_movies.shape)
df_tmdb_imdb_movies.head(3)

(424395, 10)


,title,imdb_id,budget,revenue,runtime,genres,vote_average_imdb,vote_count_imdb,popularity,release_year
0,inception,tt1375666,160000000,825532764,148,"Action, Science Fiction, Adventure",8.8,2813028,83.952,2010
1,interstellar,tt0816692,165000000,701729206,169,"Adventure, Drama, Science Fiction",8.7,2523452,140.241,2014
2,the dark knight,tt0468569,185000000,1004558444,152,"Drama, Action, Crime, Thriller",9.1,3163410,130.643,2008


Также можно заметить, что в некоторых строках, например, `budget` и `revenue`, отмечено 0. Это явный пропуск, поскольку производство фильма не может стоить 0 и принести 0 дохода.


In [237]:
cols = [
    "vote_average_imdb",
    "vote_count_imdb",
    "popularity",
    "budget",
    "revenue",
    "runtime",
]
for col in cols:
    zeros = (df_tmdb_imdb_movies[col] == 0).sum()
    print(
        f"{col}: {zeros} нулевых значений из {df_tmdb_imdb_movies[col].notna().sum()} непустых"
    )

vote_average_imdb: 0 нулевых значений из 424395 непустых
vote_count_imdb: 0 нулевых значений из 424395 непустых
popularity: 10698 нулевых значений из 424395 непустых
budget: 395854 нулевых значений из 424395 непустых
revenue: 408259 нулевых значений из 424395 непустых
runtime: 55583 нулевых значений из 424395 непустых


Избавимся от пропусков по полям `budget`, `revenue`, `runtime`, `popularity` явно указав `NAN`


In [238]:
for col in ["popularity", "budget", "revenue", "runtime"]:
    df_tmdb_imdb_movies[col] = pd.to_numeric(df_tmdb_imdb_movies[col]).replace(
        0, np.nan
    )


In [239]:
df_tmdb_imdb_movies.isna().sum()

title                     0
imdb_id                   0
budget               395854
revenue              408259
runtime               55583
genres                74669
vote_average_imdb         0
vote_count_imdb           0
popularity            10698
release_year          17274
dtype: int64

По некоторым признакам пропусков слишком много. В данный момент ничего с ними не будем делать и проведем слияние датасетов, и потом оценить результат слияния


### 2.2 Чистка `df_imdb_ids`

- Подготовим признаки с указанием года `startYear`, `endYear` - приведем к числовому формату
- Заголовки приведем к нижнему регистру


In [240]:
df_imdb_ids["startYear"] = pd.to_numeric(
    df_imdb_ids["startYear"], errors="coerce"
).astype("Int64")
df_imdb_ids["endYear"] = pd.to_numeric(df_imdb_ids["endYear"], errors="coerce").astype(
    "Int64"
)
df_imdb_ids["primary_title_lower"] = df_imdb_ids["primaryTitle"].str.lower().str.strip()

df_imdb_ids = df_imdb_ids.drop(columns=["primaryTitle"])
df_imdb_ids = df_imdb_ids.rename(
    columns={"tconst": "imdb_id", "startYear": "start_year", "endYear": "end_year"}
)

print(df_imdb_ids.shape)
df_imdb_ids.head(3)


(12475319, 4)


,imdb_id,start_year,end_year,primary_title_lower
0,tt0000001,1894,<NA>,carmencita
1,tt0000002,1892,<NA>,le clown et ses chiens
2,tt0000003,1892,<NA>,poor pierrot


## Шаг 3. Слияние датасетов


Будем поочередно мерджить исходный датасет `df_cleared` сначала с датасетом `df_imdb_ids`, затем с датасетом `df_tmdb_imdb_movies`, чтобы максимально получить новых данных.


### 3.1 Добавление `imdb_id` в исходный датасет `df_cleared`

Попробуем получить `imdb_id` из `df_imdb_ids`, чтобы затем выполнить более **точный** join с `df_tmdb_imdb_movies` по `imdb_id`, а не только по `title` и `release_year`.

Итоговый датасет `df_merged`.


In [241]:
def year_near(netflix_year, start_year, end_year):
    if pd.isna(netflix_year):
        return True
    ny = int(netflix_year)
    if pd.notna(start_year) and int(start_year) == ny:
        return True
    if pd.notna(end_year) and int(end_year) == ny:
        return True
    if (
        pd.notna(start_year)
        and pd.notna(end_year)
        and ny >= int(start_year)
        and ny <= int(end_year)
    ):
        return True
    return False


df_need_id = df_cleared[["title", "title_lower", "release_year"]].copy()

df_with_ids = pd.merge(
    df_need_id,
    df_imdb_ids,
    left_on=["title_lower"],
    right_on=["primary_title_lower"],
    how="left",
)

df_with_ids["year_match"] = df_with_ids.apply(
    lambda r: year_near(r["release_year"], r["start_year"], r["end_year"]),
    axis=1,
)

df_with_ids = (
    df_with_ids[df_with_ids["year_match"]]
    .drop(columns=["year_match", "start_year", "end_year"])
    .drop_duplicates(subset=["title", "release_year"])
)[["title", "release_year", "imdb_id"]]


df_merged = df_cleared.merge(df_with_ids, on=["title", "release_year"], how="left")


Результат слияние - всего 93 строки без подобранного `imdb_id`.
Следовательно, для таких строк для слияния сможем использовать только `title` и `release_year`


In [242]:
df_merged["imdb_id"].isna().sum()

np.int64(93)

In [243]:
df_merged.shape

(499, 5)

### 3.2 Теперь выполним слияние `df_merged` с датасетом `df_tmdb_imdb_movies` по `imdb_id`


In [244]:
def enrich_by_imdb_id(df_main, df_source, source_cols):
    df_source_sub = (
        df_source[["imdb_id"] + [c for c in source_cols if c in df_source.columns]]
        .dropna(subset=["imdb_id"])
        .drop_duplicates(subset=["imdb_id"])
        .copy()
    )

    df_with_id = df_main[df_main["imdb_id"].notna()].copy()
    df_no_id = df_main[df_main["imdb_id"].isna()].copy()

    cols_to_drop = [c for c in source_cols if c in df_with_id.columns]
    df_with_id = df_with_id.drop(columns=cols_to_drop)
    df_with_id = pd.merge(df_with_id, df_source_sub, on="imdb_id", how="left")

    return pd.concat([df_with_id, df_no_id], ignore_index=True)


cols = [
    "vote_average_imdb",
    "vote_count_imdb",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "popularity",
]
df_merged = enrich_by_imdb_id(df_merged, df_tmdb_imdb_movies, cols)


In [245]:
print(df_merged.shape)
df_merged.isna().sum()

(499, 12)


title                  0
rating                 0
release_year           0
title_lower            0
imdb_id               93
vote_average_imdb    321
vote_count_imdb      321
budget               412
revenue              411
runtime              330
genres               328
popularity           326
dtype: int64

In [246]:
df_merged.head(3)

,title,rating,release_year,title_lower,imdb_id,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,popularity
0,White Chicks,PG-13,2004,white chicks,tt0381707,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",54.851
1,Lucky Number Slevin,R,2006,lucky number slevin,tt0425210,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",21.607
2,Prison Break,TV-14,2008,prison break,tt0455275,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.3 Мерджим `df_merged` с `df_tmdb_imdb_movies` по `title` + `release_year`

Слияние будем выполнять по заголовку `title` и году выпуска контента `release_year`

Через полное слияние `matched_count` заполним пропуски


In [247]:
df_merged_copy = df_merged.copy()

df_merged = pd.merge(
    df_merged_copy,
    df_tmdb_imdb_movies,
    left_on=["title_lower", "release_year"],
    right_on=["title", "release_year"],
    how="left",
    indicator=True,
)

df_merged = df_merged.rename(
    columns={
        "title_x": "title",
        "imdb_id_x": "imdb_id",
        "vote_average_imdb_x": "vote_average_imdb",
        "vote_count_imdb_x": "vote_count_imdb",
        "budget_x": "budget",
        "revenue_x": "revenue",
        "runtime_x": "runtime",
        "genres_x": "genres",
        "popularity_x": "popularity",
    }
)

matched_count = (df_merged["_merge"] == "both").sum()

# Перезаписываем если: imdb_id пустой или имеется конфликт для следующих id
# (конфликты проверены вручную - imdb_id_y верный)
conflict_list = [
    "tt10778398",
    "tt11253872",
    "tt12853176",
    "tt10649072",
    "tt13808486",
    "tt12285944",
    "tt1604113",
    "tt10017814",
    "tt21376132",
    "tt37149812",
    "tt39028344",
    "tt5260026",
    "tt10887760",
    "tt20225680",
    "tt1794595",
    "tt18815012",
    "tt11262310",
    "tt5317732",
    "tt14009360",
    "tt2123342",
    "tt37149811",
]
overwrite_mask = df_merged["imdb_id"].isna() | (
    df_merged["imdb_id_y"].notna() & (df_merged["imdb_id"].isin(conflict_list))
)
df_merged.loc[overwrite_mask, "imdb_id"] = df_merged.loc[overwrite_mask, "imdb_id_y"]

df_merged["vote_average_imdb"] = df_merged["vote_average_imdb"].fillna(
    df_merged["vote_average_imdb_y"]
)
df_merged["vote_count_imdb"] = df_merged["vote_count_imdb"].fillna(
    df_merged["vote_count_imdb_y"]
)
df_merged["budget"] = df_merged["budget"].fillna(df_merged["budget_y"])
df_merged["revenue"] = df_merged["revenue"].fillna(df_merged["revenue_y"])
df_merged["runtime"] = df_merged["runtime"].fillna(df_merged["runtime_y"])
df_merged["genres"] = df_merged["genres"].fillna(df_merged["genres_y"])
df_merged["popularity"] = df_merged["popularity"].fillna(df_merged["popularity_y"])

df_merged = df_merged.drop(
    columns=[
        "_merge",
        "imdb_id_y",
        "title_y",
        "budget_y",
        "revenue_y",
        "runtime_y",
        "genres_y",
        "vote_average_imdb_y",
        "vote_count_imdb_y",
        "popularity_y",
    ]
)

print(f"Смерджено {matched_count} записей по title + year")
print(df_merged.shape)
df_merged.head(3)

Смерджено 190 записей по title + year
(499, 12)


,title,rating,release_year,title_lower,imdb_id,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,popularity
0,White Chicks,PG-13,2004,white chicks,tt0381707,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",54.851
1,Lucky Number Slevin,R,2006,lucky number slevin,tt0425210,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",21.607
2,Prison Break,TV-14,2008,prison break,tt0455275,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [248]:
df_merged.isna().sum()

title                  0
rating                 0
release_year           0
title_lower            0
imdb_id               83
vote_average_imdb    285
vote_count_imdb      285
budget               397
revenue              397
runtime              295
genres               292
popularity           289
dtype: int64

Проверим, что одинаковые названия но разные года выпуска смержены корректно


In [249]:
duplicated_titles_list = df_merged["title_lower"][
    df_merged.duplicated(subset=["title_lower"], keep=False)
].unique()
print(duplicated_titles_list)

<StringArray>
['skins', 'star wars: the clone wars', 'goosebumps']
Length: 3, dtype: str


In [250]:
df_tmdb_imdb_movies[
    df_tmdb_imdb_movies["title"].isin(
        ["skins", "star wars: the clone wars", "goosebumps"]
    )
].sort_values("title")

,title,imdb_id,budget,revenue,runtime,genres,vote_average_imdb,vote_count_imdb,popularity,release_year
1265,goosebumps,tt1051904,58000000.0,158162788.0,103.0,"Adventure, Horror, Comedy",6.3,102366,24.932,2015
6612,skins,tt5808778,NaN,NaN,78.0,"Drama, Comedy",6.0,7975,13.346,2017
77180,skins,tt0284494,NaN,249204.0,84.0,"Action, Drama",7.0,1679,3.880,2002
312711,skins,tt0120143,NaN,NaN,14.0,"Comedy, Science Fiction",7.2,22,0.740,1997
2354,star wars: the clone wars,tt1185834,8500000.0,68282844.0,98.0,"Animation, Action, Science Fiction, Adventure",6.0,81401,19.221,2008
286138,star wars: the clone wars,tt9313978,NaN,NaN,23.0,NaN,9.9,24021,NaN,<NA>


In [251]:
df_merged[
    df_merged["title_lower"].isin(["skins", "star wars: the clone wars", "goosebumps"])
].sort_values("title_lower")

,title,rating,release_year,title_lower,imdb_id,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,popularity
262,Goosebumps,TV-Y7,1998,goosebumps,tt0111987,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296,Goosebumps,PG,2015,goosebumps,tt1051904,6.3,102366.0,58000000.0,158162788.0,103.0,"Adventure, Horror, Comedy",24.932
93,Skins,TV-MA,2013,skins,tt0840196,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,Skins,TV-MA,2017,skins,tt5808778,6.0,7975.0,NaN,NaN,78.0,"Drama, Comedy",13.346
226,Star Wars: The Clone Wars,PG,2008,star wars: the clone wars,tt0458290,8.5,143630.0,8500000.0,68282844.0,98.0,"Animation, Action, Science Fiction, Adventure",19.221
229,Star Wars: The Clone Wars,TV-PG,2014,star wars: the clone wars,tt0458290,8.5,143630.0,NaN,NaN,NaN,NaN,NaN


## Шаг 4. Заполнение пропусков в финальном датасете


Проверим, сколько осталось пропусков в результате


In [252]:
df_merged.isna().sum()

title                  0
rating                 0
release_year           0
title_lower            0
imdb_id               83
vote_average_imdb    285
vote_count_imdb      285
budget               397
revenue              397
runtime              295
genres               292
popularity           289
dtype: int64

Таким образом, в некоторых полях по-прежнему имеются пропуски.


### 4.1 Заполнение пропусков в `genres`, `runtime` и добавление нового поля `titleType`

Снова обратимся к [датасету](https://developer.imdb.com/non-commercial-datasets/?utm_source=chatgpt.com#titlebasicstsvgz) с официального сайта IMDB, который содержит информацию о жанрах, времени контента и типе контента

Так как ранее по нему мы обогатились `imdb_id`, слияние будем выолнять по этому признаку


In [253]:
df_imdb = pd.read_csv(
    "title.basics.tsv",
    sep="\t",
    na_values=["\\N"],
    usecols=["tconst", "titleType", "runtimeMinutes", "genres"],
    low_memory=False,
)

In [254]:
df_imdb.head(3)

,tconst,titleType,runtimeMinutes,genres
0,tt0000001,short,1,"Documentary,Short"
1,tt0000002,short,5,"Animation,Short"
2,tt0000003,short,5,"Animation,Comedy,Romance"


Подготовим данные к слиянию

- Переименуем колонки
- Колонку с жанрами отформатируем как в итоговом датасете


In [255]:
df_imdb["runtimeMinutes"] = pd.to_numeric(
    df_imdb["runtimeMinutes"], errors="coerce"
).replace(0, np.nan)

df_imdb = df_imdb.rename(
    columns={
        "tconst": "imdb_id",
        "titleType": "content_type",
        "runtimeMinutes": "runtime",
    }
)

df_imdb["genres"] = df_imdb["genres"].str.replace(",", ", ")

print(df_imdb.shape)
df_imdb.head(5)

(12475319, 4)


,imdb_id,content_type,runtime,genres
0,tt0000001,short,1.0,"Documentary, Short"
1,tt0000002,short,5.0,"Animation, Short"
2,tt0000003,short,5.0,"Animation, Comedy, Romance"
3,tt0000004,short,12.0,"Animation, Short"
4,tt0000005,short,1.0,Short


In [256]:
df_imdb.isna().sum()

imdb_id               0
content_type          0
runtime         7996070
genres           536138
dtype: int64

Выполним слияние по `imdb_id`


In [257]:
df_merged = pd.merge(
    df_merged,
    df_imdb,
    left_on="imdb_id",
    right_on="imdb_id",
    how="left",
    indicator=True,
)

df_merged = df_merged.rename(
    columns={
        "runtime_x": "runtime",
        "genres_x": "genres",
    }
)

matched_count = (df_merged["_merge"] == "both").sum()

df_merged["runtime"] = df_merged["runtime"].fillna(df_merged["runtime_y"])
df_merged["genres"] = df_merged["genres"].fillna(df_merged["genres_y"])

df_merged = df_merged.drop(
    columns=[
        "_merge",
        "runtime_y",
        "genres_y",
    ]
)

print(f"Смерджено {matched_count} записей")
print(df_merged.shape)
df_merged.head(3)

Смерджено 416 записей
(499, 13)


,title,rating,release_year,title_lower,imdb_id,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,popularity,content_type
0,White Chicks,PG-13,2004,white chicks,tt0381707,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",54.851,movie
1,Lucky Number Slevin,R,2006,lucky number slevin,tt0425210,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",21.607,movie
2,Prison Break,TV-14,2008,prison break,tt0455275,NaN,NaN,NaN,NaN,44.0,"Action, Crime, Drama",NaN,tvSeries


In [258]:
df_merged.isna().sum()

title                  0
rating                 0
release_year           0
title_lower            0
imdb_id               83
vote_average_imdb    285
vote_count_imdb      285
budget               397
revenue              397
runtime              105
genres                83
popularity           289
content_type          83
dtype: int64

Благодаря датасету IMDB удалось существенно сократить количество пропусков в полях `genres`, `runtime`


Оставшиеся пропуски в колонках планируем заполнить с помощью парсинга в следующей части анализа.


In [259]:
df_merged.to_csv("netflix_enriched_2.csv", index=False)